# EXP_020E — Image Ablation: SigLIP + XLM-R + Concat + MSE
**Phase 2 | Image Backbone Ablation**
Research question: Does SigLIP provide better visual features than ConvNeXt for noisy restaurant review images?
- Image model: `google/siglip-base-patch16-256` | Text: `xlm-roberta-base` (from EXP_010)
- Fusion: Concatenation + MLP | Loss: MSE | Seed: 42 | AMP: enabled
> ⚠️ Prerequisite: EXP_010 must be completed and saved to Drive.

### STEP 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### STEP 2: Clone source code and install dependencies

In [ ]:
!git clone https://github.com/lechihoang/SE365.git
%cd SE365
!pip install -r requirements.txt -q

Cloning into 'SE365'...
remote: Enumerating objects: 13330, done.
remote: Counting objects: 100% (201/201), done.
remote: Compressing objects: 100% (110/110), done.
remote: Total 13330 (delta 143), reused 146 (delta 91), pack-reused 13129 (from 1)
Receiving objects: 100% (13330/13330), 873.19 MiB | 18.87 MiB/s, done.
Resolving deltas: 100% (381/381), done.
/content/SE365


### STEP 3: Download and extract data

In [ ]:
!rm -rf ./data
!gdown --id 11WoeUn2visKtGN5oOX9c2I6Grz3P88vD -O data.zip
!unzip -q data.zip
!rm data.zip
!ls -la ./data

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=11WoeUn2visKtGN5oOX9c2I6Grz3P88vD
From (redirected): https://drive.google.com/uc?id=11WoeUn2visKtGN5oOX9c2I6Grz3P88vD&confirm=t&uuid=efe11f4f-6dad-471c-85fa-47adf66b889b
To: /content/SE365/data.zip
100% 4.02G/4.02G [00:39<00:00, 101MB/s]
total 1344
drwxr-xr-x  4 root root    4096 Jun 16 09:21 .
drwxr-xr-x 11 root root    4096 Jun 23 07:45 ..
drwxr-xr-x  2 root root 1359872 Jun 16 09:59 image
drwxr-xr-x  2 root root    4096 Jun 16 09:21 text


### STEP 4: Configure paths

In [ ]:
import os
DRIVE_ROOT = '/content/drive/MyDrive/SE365'  # ✏️ Change if needed
EXP_ID = 'EXP_020E_siglip_xlmr_concat_mse'
DRIVE_EXP_PATH = f'{DRIVE_ROOT}/experiments/{EXP_ID}'
os.makedirs(DRIVE_EXP_PATH, exist_ok=True)
print(f'Artifacts will be saved to: {DRIVE_EXP_PATH}')

Artifacts will be saved to: /content/drive/MyDrive/SE365/experiments/EXP_020E_siglip_xlmr_concat_mse


### STEP 5: Load pretrained text weights from EXP_010


In [ ]:
import os, shutil
os.makedirs('./checkpoints', exist_ok=True)
shutil.copy(f'{DRIVE_ROOT}/experiments/EXP_010_text_only_xlmr_mse/best_model_train_text.pth', './checkpoints/best_model_train_text.pth')
print('Loaded text weights from EXP_010')


Loaded text weights from EXP_010


### STEP 6: Pre-train Image Branch
Fine-tune the image model before fusion to align its features with the score prediction task.

In [ ]:
!python main.py \
  --mode train_image \
  --image_model_name vit_base_patch16_siglip_256 \
  --epochs 20 \
  --batch_size 16 \
  --lr 1e-5 \
  --loss_fn mse \
  --seed 42 \
  --use_amp \
  --exp_dir ./experiments

====== MODE: TRAIN_IMAGE ======
Using device: cuda
Seed: 42 | Experiment: EXP_000
config.json: 100% 615/615 [00:00<00:00, 2.30MB/s]
tokenizer_config.json: 100% 25.0/25.0 [00:00<00:00, 116kB/s]
sentencepiece.bpe.model: 100% 5.07M/5.07M [00:01<00:00, 3.61MB/s]
tokenizer.json: 100% 9.10M/9.10M [00:01<00:00, 6.66MB/s]
preprocessor_config.json: 100% 368/368 [00:00<00:00, 1.53MB/s]
Loaded HuggingFace processor for vit_base_patch16_siglip_256
model.safetensors: 100% 372M/372M [00:06<00:00, 57.6MB/s]
/content/SE365/Trainer.py:82: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(enabled=self.use_amp)
2026-06-23 07:47:41 ====== START: 2026-06-23T07:47:41.041845 ======
2026-06-23 07:47:41 Experiment: EXP_000 | Mode: train_image | exp_dir: ./experiments/EXP_000
Epoch 1/20:   0% 0/300 [00:00<?, ?it/s]/content/SE365/Trainer.py:121: FutureWarning: `torch.cuda.amp.autocast(args...)` 

### STEP 7: Train Fusion Model


In [ ]:
!python main.py \
  --mode train_fusion \
  --text_model_name xlm-roberta-base \
  --image_model_name vit_base_patch16_siglip_256 \
  --epochs 15 \
  --batch_size 16 \
  --lr 1e-5 \
  --grad_accum_steps 2 \
  --patience 5 \
  --loss_fn mse \
  --unfreeze_text_layers 1 \
  --unfreeze_image_layers 1 \
  --seed 42 \
  --use_amp \
  --exp_id EXP_020E_siglip_xlmr_concat_mse \
  --exp_dir ./experiments

====== MODE: TRAIN_FUSION ======
Using device: cuda
Seed: 42 | Experiment: EXP_020E_siglip_xlmr_concat_mse
Loaded HuggingFace processor for vit_base_patch16_siglip_256
model.safetensors: 100% 1.12G/1.12G [00:03<00:00, 328MB/s]
Loading weights: 100% 199/199 [00:00<00:00, 4447.26it/s]
[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/content/SE365/Trainer.py:82: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(enabled=self.us

### STEP 8: Save to Drive + print metrics


In [ ]:
import json
!cp -r ./experiments/$EXP_ID/* $DRIVE_EXP_PATH/

with open(f'./experiments/{EXP_ID}/metrics.json') as f:
    m = json.load(f)

print(f'\n=== {EXP_ID} Results ===')
print(f"Loss (val)   : {m['loss']:.4f}")
print()
print("             MAE      RMSE      R2")
print(f"  food     : {m['mae_food']:.4f}   {m['rmse_food']:.4f}   {m['r2_food']:.4f}")
print(f"  price    : {m['mae_price']:.4f}   {m['rmse_price']:.4f}   {m['r2_price']:.4f}")
print(f"  atmos    : {m['mae_atmos']:.4f}   {m['rmse_atmos']:.4f}   {m['r2_atmos']:.4f}")
print(f"  service  : {m['mae_service']:.4f}   {m['rmse_service']:.4f}   {m['r2_service']:.4f}")
print(f"  overall  : {m['mae_overall']:.4f}   {m['rmse_overall']:.4f}   {m['r2_overall']:.4f}")
print()
print(f"  mean_mae   : {m['mean_mae']:.4f}")
print(f"  aspect_mae : {m['aspect_mae']:.4f}")
print(f"  overall_mae: {m['overall_mae']:.4f}")


=== EXP_020E_siglip_xlmr_concat_mse Results ===
Loss (val)   : 2.7929

             MAE      RMSE      R2
  food     : 1.2522   1.7402   0.4246
  price    : 1.2804   1.7489   0.3152
  atmos    : 1.2357   1.6307   0.3148
  service  : 1.3095   1.7634   0.3935
  overall  : 1.0703   1.4669   0.4715

  mean_mae   : 1.2296
  aspect_mae : 1.2695
  overall_mae: 1.0703
